# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,32.0,0.027879,-0.589744,0.205128,26,6
1,CTR vs PD,PES_RG,26.5,0.013779,-0.660256,0.169872,26,6
2,CTR vs PD,PES_BY,33.0,0.031644,-0.576923,0.211538,26,6
3,CTR vs PD,VKS_MajRad,35.0,0.037778,-0.551282,0.224359,26,6
4,CTR vs PD,VKS_MinRad,33.0,0.028917,-0.576923,0.211538,26,6
5,CTR vs PD,VKS_Angle,76.0,0.943789,-0.025641,0.487179,26,6
6,HC vs protan,TES,6.0,0.000030,-0.948718,0.025641,26,9
7,HC vs protan,PES_RG,0.0,0.000011,-1.000000,0.000000,26,9
8,HC vs protan,PES_BY,55.0,0.020260,-0.529915,0.235043,26,9
9,HC vs protan,VKS_MajRad,7.0,0.000036,-0.940171,0.029915,26,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,7.0,0.000005,-0.932692,0.033654,26,8
15,HC vs deutan,VKS_MajRad,9.0,0.000011,-0.913462,0.043269,26,8
7,HC vs protan,PES_RG,0.0,0.000011,-1.000000,0.000000,26,9
13,HC vs deutan,PES_RG,0.0,0.000026,-1.000000,0.000000,26,8
6,HC vs protan,TES,6.0,0.000030,-0.948718,0.025641,26,9
9,HC vs protan,VKS_MajRad,7.0,0.000036,-0.940171,0.029915,26,9
12,HC vs deutan,TES,8.0,0.000104,-0.923077,0.038462,26,8
10,HC vs protan,VKS_MinRad,16.0,0.000149,-0.863248,0.068376,26,9
14,HC vs deutan,PES_BY,25.0,0.001435,-0.759615,0.120192,26,8
11,HC vs protan,VKS_Angle,193.0,0.004378,0.649573,0.824786,26,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,7.0,0.000005,-0.932692,0.033654,26,8,0.000030,True
15,HC vs deutan,VKS_MajRad,9.0,0.000011,-0.913462,0.043269,26,8,0.000053,True
7,HC vs protan,PES_RG,0.0,0.000011,-1.000000,0.000000,26,9,0.000066,True
13,HC vs deutan,PES_RG,0.0,0.000026,-1.000000,0.000000,26,8,0.000105,True
9,HC vs protan,VKS_MajRad,7.0,0.000036,-0.940171,0.029915,26,9,0.000150,True
6,HC vs protan,TES,6.0,0.000030,-0.948718,0.025641,26,9,0.000150,True
12,HC vs deutan,TES,8.0,0.000104,-0.923077,0.038462,26,8,0.000313,True
10,HC vs protan,VKS_MinRad,16.0,0.000149,-0.863248,0.068376,26,9,0.000446,True
14,HC vs deutan,PES_BY,25.0,0.001435,-0.759615,0.120192,26,8,0.002869,True
11,HC vs protan,VKS_Angle,193.0,0.004378,0.649573,0.824786,26,9,0.008756,True
